In [3]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

# ==========================================
# 1. INVESTOR PROFILE INPUTS (From Model 1)
# ==========================================
BRI = 0.70          # Behavioural Risk Index (0.0 to 1.0)
alpha = 0.50        # Trend Willingness coefficient (0.2 = Low, 0.5 = Med, 0.8 = High)
F = 1000000         # Total available funds in LKR
RC = 0.20           # Risk Capacity (Max acceptable annualized volatility, e.g., 20%)[cite: 1]

# Derive Risk Penalty Lambda (λ)[cite: 1]
# Higher BRI -> Lower Risk Aversion Penalty[cite: 1]
lambda_penalty = 1.0 - BRI

# ==========================================
# 2. SECTOR INPUTS (From Model 2)
# ==========================================
sector_data = pd.DataFrame({
    'Sector': ['Telecommunication', 'Retail', 'Transportation', 'Banking', 'Food & Beverage'],
    'Expected_Return': [0.12, 0.15, 0.18, 0.10, 0.08],
    'Trend_Score': [0.60, 0.20, -0.40, 0.80, 0.10]  # T_i between -1 and +1[cite: 1]
})

In [4]:
# Calculate Trend-Adjusted Sector Return Score[cite: 1]
sector_data['Adjusted_Score'] = sector_data['Expected_Return'] * (1 + alpha * sector_data['Trend_Score'])

# Sort sectors and pick top K sectors (e.g., Top 3)
TOP_K_SECTORS = 3
selected_sectors = sector_data.sort_values(by='Adjusted_Score', ascending=False).head(TOP_K_SECTORS)

print("--- SELECTED TOP SECTORS ---")
print(selected_sectors[['Sector', 'Expected_Return', 'Trend_Score', 'Adjusted_Score']])

--- SELECTED TOP SECTORS ---
              Sector  Expected_Return  Trend_Score  Adjusted_Score
1             Retail             0.15          0.2           0.165
0  Telecommunication             0.12          0.6           0.156
2     Transportation             0.18         -0.4           0.144


In [5]:
# =========================================================
# MOCK CSE STOCK UNIVERSE (Replace with real CSE DataFrames)
# =========================================================
stock_universe = pd.DataFrame({
    'Ticker': ['DIAL.N0000', 'SLTL.N0000', 'JKH.N0000', 'CARG.N0000', 'COMB.N0000'],
    'Sector': ['Telecommunication', 'Telecommunication', 'Transportation', 'Retail', 'Banking'],
    'Expected_Return': [0.11, 0.13, 0.19, 0.14, 0.09],
    'Beta': [0.85, 1.10, 1.25, 0.90, 1.05],
    'Public_Float_Pct': [0.25, 0.30, 0.40, 0.18, 0.35]  # From Public Holding dataset
})

# Filter 1: Retain stocks only from the selected sectors
filtered_stocks = stock_universe[stock_universe['Sector'].isin(selected_sectors['Sector'])].copy()

# Filter 2: Liquidity Constraint (e.g., minimum 20% public holding float)
MIN_FLOAT = 0.20
filtered_stocks = filtered_stocks[filtered_stocks['Public_Float_Pct'] >= MIN_FLOAT].copy()

# Merge Sector Trend Scores (T_i) into stock dataset
filtered_stocks = filtered_stocks.merge(selected_sectors[['Sector', 'Trend_Score']], on='Sector')

# Calculate Beta-Adjusted Stock Expected Return
# R_adj = R * (1 + alpha * Beta * Trend_Score)
filtered_stocks['R_adj'] = filtered_stocks['Expected_Return'] * (
    1 + alpha * filtered_stocks['Beta'] * filtered_stocks['Trend_Score']
)

n_stocks = len(filtered_stocks)
stock_tickers = filtered_stocks['Ticker'].tolist()
R_adj_vector = filtered_stocks['R_adj'].values

print("\n--- QUALIFIED STOCKS FOR OPTIMIZATION ---")
print(filtered_stocks[['Ticker', 'Sector', 'Beta', 'Expected_Return', 'R_adj']])


--- QUALIFIED STOCKS FOR OPTIMIZATION ---
       Ticker             Sector  Beta  Expected_Return    R_adj
0  DIAL.N0000  Telecommunication  0.85             0.11  0.13805
1  SLTL.N0000  Telecommunication  1.10             0.13  0.17290
2   JKH.N0000     Transportation  1.25             0.19  0.14250


In [6]:
# In practice: Read from CSE Daily Share Price Lists, compute log returns, and derive cov matrix.
# Here we simulate an N x N covariance matrix for the qualified stocks:
np.random.seed(42)
mock_returns = np.random.normal(0.0005, 0.015, (252, n_stocks))  # 252 trading days
cov_matrix = np.cov(mock_returns, rowvar=False) * 252           # Annualized covariance matrix

In [7]:
# 1. Objective Function (Minimized as negative utility)[cite: 1]
def objective_function(weights, R_adj, cov_mat, lam):
    portfolio_return = np.dot(weights, R_adj)
    portfolio_variance = np.dot(weights.T, np.dot(cov_mat, weights))

    # We want to maximize: Return - λ * Variance[cite: 1]
    # Minimize: -(Return - λ * Variance)
    utility = portfolio_return - lam * portfolio_variance
    return -utility

# 2. Portfolio Risk Utility Function (for constraint checking)
def get_portfolio_volatility(weights, cov_mat):
    return np.sqrt(np.dot(weights.T, np.dot(cov_mat, weights)))

# 3. Constraints Setup[cite: 1]
constraints = [
    # Constraint 1: Sum of weights = 1 (Full Investment)[cite: 1]
    {'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0},

    # Constraint 2: Portfolio Volatility <= Investor Risk Capacity (RC)[cite: 1]
    {'type': 'ineq', 'fun': lambda w: RC - get_portfolio_volatility(w, cov_matrix)}
]

# 4. Bounds: No short selling (w_k >= 0) and stock concentration limits (e.g., max 40% in one stock)[cite: 1]
bounds = tuple((0.0, 0.40) for _ in range(n_stocks))

# 5. Initial Guess (Equal weighting)
initial_weights = np.ones(n_stocks) / n_stocks

# 6. Execute SLSQP Optimization Algorithm[cite: 1]
opt_result = minimize(
    fun=objective_function,
    x0=initial_weights,
    args=(R_adj_vector, cov_matrix, lambda_penalty),
    method='SLSQP',  # SLSQP algorithm as specified[cite: 1]
    bounds=bounds,
    constraints=constraints
)

# Extract optimal weights
optimal_weights = opt_result.x

In [8]:
# Calculate metrics
filtered_stocks['Optimal_Weight'] = optimal_weights
filtered_stocks['Monetary_Allocation_LKR'] = optimal_weights * F

expected_portfolio_return = np.dot(optimal_weights, R_adj_vector)
expected_portfolio_volatility = get_portfolio_volatility(optimal_weights, cov_matrix)

print("\n================ FINAL OPTIMAL PORTFOLIO ================")
print(filtered_stocks[['Ticker', 'Sector', 'Optimal_Weight', 'Monetary_Allocation_LKR']])
print("---------------------------------------------------------")
print(f"Total Available Capital : LKR {F:,.2f}")
print(f"Expected Portfolio Return: {expected_portfolio_return * 100:.2f}%")
print(f"Expected Portfolio Risk  : {expected_portfolio_volatility * 100:.2f}% (Limit: {RC * 100:.2f}%)")
print(f"Applied Risk Penalty (λ) : {lambda_penalty:.2f} (Based on BRI={BRI})")


================ FINAL OPTIMAL PORTFOLIO ================
       Ticker             Sector  Optimal_Weight  Monetary_Allocation_LKR
0  DIAL.N0000  Telecommunication        0.255324            255323.857077
1  SLTL.N0000  Telecommunication        0.400000            400000.000000
2   JKH.N0000     Transportation        0.344676            344676.142923
---------------------------------------------------------
Total Available Capital : LKR 1,000,000.00
Expected Portfolio Return: 15.35%
Expected Portfolio Risk  : 13.54% (Limit: 20.00%)
Applied Risk Penalty (λ) : 0.30 (Based on BRI=0.7)
